In [21]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [22]:
DATA_DIR = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/"
ANN_DIR = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations"

In [23]:
PHASES = [
    'tPB2','tPNa','tPNf','t2','t3','t4','t5','t6','t7','t8','t9+',
    'tM','tSB','tB','tEB','tHB'
]

label_map = {p:i for i,p in enumerate(PHASES)}

In [24]:
def build_dataframe(sample_rate=6):
    data = []

    for file in tqdm(os.listdir(ANN_DIR)):
        if not file.endswith(".csv"):
            continue
        
        embryo_id = file.replace("_phases.csv","")
        ann_path = os.path.join(ANN_DIR, file)
        img_folder = os.path.join(DATA_DIR, embryo_id)

        if not os.path.exists(img_folder):
            continue

        images = sorted(os.listdir(img_folder))
        if len(images) == 0:
            continue

        df = pd.read_csv(ann_path, header=None)
        df.columns = ["phase","start","end"]

        for _, row in df.iterrows():
            label = label_map[row["phase"]]

            for frame in range(row["start"], row["end"], sample_rate):
                if frame < len(images):
                    img_path = os.path.join(img_folder, images[frame])
                    data.append([img_path, label, embryo_id])

    return pd.DataFrame(data, columns=["image","label","embryo"])

In [25]:
df = build_dataframe()

print("\nTotal samples:", len(df))
print(df.head())

100%|██████████| 704/704 [00:02<00:00, 310.81it/s]


Total samples: 52179
                                               image  label   embryo
0  /kaggle/input/datasets/abhishekbuddiga06/embry...      0  BC750-7
1  /kaggle/input/datasets/abhishekbuddiga06/embry...      0  BC750-7
2  /kaggle/input/datasets/abhishekbuddiga06/embry...      1  BC750-7
3  /kaggle/input/datasets/abhishekbuddiga06/embry...      1  BC750-7
4  /kaggle/input/datasets/abhishekbuddiga06/embry...      1  BC750-7


In [26]:
embryos = df["embryo"].unique()

train_ids, temp_ids = train_test_split(embryos, test_size=0.3, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_df = df[df.embryo.isin(train_ids)]
val_df = df[df.embryo.isin(val_ids)]
test_df = df[df.embryo.isin(test_ids)]

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


Split sizes:
Train: 36415
Val: 7867
Test: 7897


In [ ]:
print("\nBefore merge:")
print(train_df["label"].value_counts().sort_index())

# Merge class 15 → 14
train_df.loc[train_df["label"] == 15, "label"] = 14
val_df.loc[val_df["label"] == 15, "label"] = 14
test_df.loc[test_df["label"] == 15, "label"] = 14

num_classes = 15

print("\nAfter merge:")
print(train_df["label"].value_counts().sort_index())


Before merge:
label
0     1177
1     5167
2      956
3     3507
4      700
5     3569
6     1102
7     1029
8     1202
9     3941
10    6285
11    1982
12    2125
13    1243
14    2414
15      16
Name: count, dtype: int64

After merge:
label
0     1177
1     5167
2      956
3     3507
4      700
5     3569
6     1102
7     1029
8     1202
9     3941
10    6285
11    1982
12    2125
13    1243
14    2430
Name: count, dtype: int64


In [28]:
classes = np.arange(num_classes)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df["label"].values
)

class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print("\nClass weights:")
for i, w in enumerate(weights):
    print(f"Class {i}: {w:.4f}")


Class weights:
Class 0: 2.0626
Class 1: 0.4698
Class 2: 2.5394
Class 3: 0.6922
Class 4: 3.4681
Class 5: 0.6802
Class 6: 2.2030
Class 7: 2.3592
Class 8: 2.0197
Class 9: 0.6160
Class 10: 0.3863
Class 11: 1.2249
Class 12: 1.1424
Class 13: 1.9531
Class 14: 0.9990


In [29]:
class EmbryoDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        for _ in range(5):
            try:
                row = self.df.iloc[idx]
                img = Image.open(row["image"]).convert("RGB")
                img = img.resize((256,256))

                if self.transform:
                    img = self.transform(img)

                label = torch.tensor(row["label"], dtype=torch.long)
                return img, label

            except:
                idx = np.random.randint(0, len(self.df))

        return torch.zeros(3,224,224), torch.tensor(0)

In [ ]:
# TRANSFORMS

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.CenterCrop(224),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
    transforms.CenterCrop(224)
])

In [31]:
train_loader = DataLoader(EmbryoDataset(train_df, train_transform), batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(EmbryoDataset(val_df, test_transform), batch_size=32, num_workers=4)
test_loader  = DataLoader(EmbryoDataset(test_df, test_transform), batch_size=32, num_workers=4)

In [32]:
from torchvision.models import *

def get_model(name):

    if name == "mobilenet":
        model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.last_channel, num_classes))

    elif name == "vgg16":
        model = vgg16(weights=VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(4096, num_classes)

    elif name == "vgg19":
        model = vgg19(weights=VGG19_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(4096, num_classes)

    return model.to(DEVICE)

In [33]:
class FinalOrdinalLoss(nn.Module):
    def __init__(self, alpha, num_classes, lambda_ord=0.5, gamma=2.0):
        super().__init__()

        self.ce = nn.CrossEntropyLoss(weight=alpha, label_smoothing=0.1)
        self.lambda_ord = lambda_ord
        self.gamma = gamma

        self.register_buffer("classes", torch.arange(num_classes).view(1, -1))

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)

        # CE
        ce = self.ce(logits, targets)

        # p_y
        p_y = probs[torch.arange(probs.size(0), device=logits.device), targets]

        # focal gate
        focal = (1.0 - p_y) ** self.gamma

        # CDF
        cdf_pred = torch.cumsum(probs, dim=1)
        cdf_true = (self.classes.to(logits.device) >= targets.unsqueeze(1)).float()

        ord_loss = ((cdf_pred - cdf_true) ** 2).mean(dim=1)

        total_loss = ce + self.lambda_ord * (focal * ord_loss).mean()

        return total_loss

In [34]:
print("""
LOSS FUNCTION:
- CrossEntropy → classification
- CDF Loss → enforces ordinal structure
- Focal gate → focuses on hard samples
""")


LOSS FUNCTION:
- CrossEntropy → classification
- CDF Loss → enforces ordinal structure
- Focal gate → focuses on hard samples



In [ ]:
def train(model, loader, optimizer, loss_fn):
    model.train()

    total_loss = 0
    all_preds, all_targets = [], []

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        out = model(imgs)
        loss = loss_fn(out, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(out, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

        if i % 100 == 0:
            batch_acc = (preds == labels).float().mean().item()
            print(f"Batch {i}/{len(loader)} | Loss: {loss.item():.4f} | Acc: {batch_acc:.4f}")

    epoch_acc = accuracy_score(all_targets, all_preds)

    return total_loss / len(loader), epoch_acc


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)

            p = torch.argmax(outputs, dim=1).cpu().numpy()
            preds.extend(p)
            targets.extend(labels.numpy())

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="weighted")

    return acc, f1

In [ ]:
results = {}

for name in ["mobilenet", "vgg16", "vgg19"]:

    print(f"\n Training {name.upper()}")

    model = get_model(name)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    loss_fn = FinalOrdinalLoss(
        alpha=class_weights,
        num_classes=num_classes
    ).to(DEVICE)

    for epoch in range(5):

        train_loss, train_acc = train(model, train_loader, optimizer, loss_fn)
        val_acc, val_f1 = evaluate(model, val_loader)

        print(f"""
==============================
Epoch {epoch}
Train Loss: {train_loss:.4f}
Train Accuracy: {train_acc:.4f}
Val Accuracy: {val_acc:.4f}
Val F1: {val_f1:.4f}
==============================
""")

    test_acc, test_f1 = evaluate(model, test_loader)
    results[name] = (test_acc, test_f1)

    print(f" FINAL {name.upper()} → Acc: {test_acc:.4f}, F1: {test_f1:.4f}")


🚀 Training MOBILENET
Batch 0/1138 | Loss: 2.9624 | Acc: 0.0312
Batch 100/1138 | Loss: 2.7965 | Acc: 0.1250
Batch 200/1138 | Loss: 2.4995 | Acc: 0.3125
Batch 300/1138 | Loss: 2.3729 | Acc: 0.2812
Batch 400/1138 | Loss: 2.3044 | Acc: 0.2188
Batch 500/1138 | Loss: 2.1523 | Acc: 0.2812
Batch 600/1138 | Loss: 2.4230 | Acc: 0.1250
Batch 700/1138 | Loss: 2.1303 | Acc: 0.3125
Batch 800/1138 | Loss: 2.4586 | Acc: 0.2188
Batch 900/1138 | Loss: 2.2495 | Acc: 0.2812
Batch 1000/1138 | Loss: 2.1362 | Acc: 0.5000
Batch 1100/1138 | Loss: 2.1387 | Acc: 0.2812

Epoch 0
Train Loss: 2.4071
Train Accuracy: 0.2555
Val Accuracy: 0.2692
Val F1: 0.2734

Batch 0/1138 | Loss: 2.0579 | Acc: 0.4062
Batch 100/1138 | Loss: 2.1493 | Acc: 0.3125
Batch 200/1138 | Loss: 2.2045 | Acc: 0.1875
Batch 300/1138 | Loss: 2.1532 | Acc: 0.2500
Batch 400/1138 | Loss: 1.9941 | Acc: 0.3438
Batch 500/1138 | Loss: 2.1396 | Acc: 0.2812
Batch 600/1138 | Loss: 2.2057 | Acc: 0.2500
Batch 700/1138 | Loss: 2.1754 | Acc: 0.3125
Batch 800/11

100%|██████████| 528M/528M [00:02<00:00, 210MB/s] 


Batch 0/1138 | Loss: 3.1347 | Acc: 0.0000
Batch 100/1138 | Loss: 2.7172 | Acc: 0.1875
Batch 200/1138 | Loss: 2.4579 | Acc: 0.2188
Batch 300/1138 | Loss: 2.3102 | Acc: 0.2812
Batch 400/1138 | Loss: 2.4467 | Acc: 0.2812
Batch 500/1138 | Loss: 2.2114 | Acc: 0.2500
Batch 600/1138 | Loss: 2.3343 | Acc: 0.2188
Batch 700/1138 | Loss: 2.3933 | Acc: 0.1875
Batch 800/1138 | Loss: 2.2156 | Acc: 0.2500
Batch 900/1138 | Loss: 2.3548 | Acc: 0.1875
Batch 1000/1138 | Loss: 2.2679 | Acc: 0.3750
Batch 1100/1138 | Loss: 2.2728 | Acc: 0.3125

Epoch 0
Train Loss: 2.4320
Train Accuracy: 0.2414
Val Accuracy: 0.2203
Val F1: 0.2133

Batch 0/1138 | Loss: 2.4384 | Acc: 0.1875
Batch 100/1138 | Loss: 2.2129 | Acc: 0.3438
Batch 200/1138 | Loss: 2.5460 | Acc: 0.2188
Batch 300/1138 | Loss: 2.0059 | Acc: 0.4375
Batch 400/1138 | Loss: 2.0941 | Acc: 0.1875
Batch 500/1138 | Loss: 2.2040 | Acc: 0.3125
Batch 600/1138 | Loss: 2.6674 | Acc: 0.3125
Batch 700/1138 | Loss: 2.2558 | Acc: 0.3438
Batch 800/1138 | Loss: 2.4284 | Ac

100%|██████████| 548M/548M [00:02<00:00, 243MB/s] 


Batch 0/1138 | Loss: 3.0026 | Acc: 0.0000
Batch 100/1138 | Loss: 2.5861 | Acc: 0.2188
Batch 200/1138 | Loss: 2.3689 | Acc: 0.1250
Batch 300/1138 | Loss: 2.3855 | Acc: 0.1562
Batch 400/1138 | Loss: 2.4432 | Acc: 0.3438
Batch 500/1138 | Loss: 2.2895 | Acc: 0.2188
Batch 600/1138 | Loss: 2.3265 | Acc: 0.3125
Batch 700/1138 | Loss: 2.4774 | Acc: 0.1875
Batch 800/1138 | Loss: 2.5588 | Acc: 0.3125
Batch 900/1138 | Loss: 2.2890 | Acc: 0.2812
Batch 1000/1138 | Loss: 2.0698 | Acc: 0.4062
Batch 1100/1138 | Loss: 2.0164 | Acc: 0.3125

Epoch 0
Train Loss: 2.4331
Train Accuracy: 0.2366
Val Accuracy: 0.2161
Val F1: 0.2089

Batch 0/1138 | Loss: 2.3469 | Acc: 0.2500
Batch 100/1138 | Loss: 2.2856 | Acc: 0.2500
Batch 200/1138 | Loss: 2.2405 | Acc: 0.3125
Batch 300/1138 | Loss: 2.4792 | Acc: 0.1875
Batch 400/1138 | Loss: 2.1279 | Acc: 0.2812
Batch 500/1138 | Loss: 2.5570 | Acc: 0.3125
Batch 600/1138 | Loss: 2.1496 | Acc: 0.2188
Batch 700/1138 | Loss: 2.3003 | Acc: 0.2812
Batch 800/1138 | Loss: 2.2089 | Ac

In [ ]:
from torchvision.models import inception_v3, Inception_V3_Weights

print("\n Training INCEPTION")

# TRANSFORMS (INCEPTION)
inception_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# DATALOADERS
train_loader_inc = DataLoader(
    EmbryoDataset(train_df, inception_transform),
    batch_size=16,
    shuffle=True,
    num_workers=4,
    drop_last=True,
    pin_memory=True
)

val_loader_inc = DataLoader(
    EmbryoDataset(val_df, inception_transform),
    batch_size=16,
    num_workers=4,
    drop_last=True,
    pin_memory=True
)

test_loader_inc = DataLoader(
    EmbryoDataset(test_df, inception_transform),
    batch_size=16,
    num_workers=4,
    drop_last=True,
    pin_memory=True
)

# MODEL
model = inception_v3(weights=Inception_V3_Weights.DEFAULT)

# MAIN HEAD
model.fc = nn.Linear(model.fc.in_features, num_classes)

# AUX HEAD
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)

model = model.to(DEVICE)

# OPTIMIZER + SCHEDULER
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=1, factor=0.5
)

loss_fn = FinalOrdinalLoss(class_weights, num_classes)

# TRAIN FUNCTION
def train_inception(model, loader):
    model.train()
    total_loss = 0

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(imgs)

        # AUX handling
        if isinstance(outputs, tuple):
            main_out, aux_out = outputs
            loss1 = loss_fn(main_out, labels)
            loss2 = loss_fn(aux_out, labels)
            loss = loss1 + 0.4 * loss2
        else:
            loss = loss_fn(outputs, labels)

        loss.backward()

        # Stability improvement
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Batch {i}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)

# EVALUATE FUNCTION
def eval_inception(model, loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)

            outputs = model(imgs)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            p = torch.argmax(outputs, dim=1).cpu().numpy()
            preds.extend(p)
            targets.extend(labels.numpy())

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="weighted")

    return acc, f1

# TRAIN LOOP
for epoch in range(5):
    loss = train_inception(model, train_loader_inc)
    acc, f1 = eval_inception(model, val_loader_inc)

    # Print LR
    current_lr = optimizer.param_groups[0]['lr']

    print(f"""
==============================
Inception Epoch {epoch}
Loss: {loss:.4f}
Val Accuracy: {acc:.4f}
Val F1: {f1:.4f}
LR: {current_lr:.6f}
==============================
""")

    scheduler.step(acc)

# FINAL TEST
acc, f1 = eval_inception(model, test_loader_inc)

print(f"\n INCEPTION FINAL → Acc: {acc:.4f}, F1: {f1:.4f}")


 Training INCEPTION
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 240MB/s] 


Batch 0/2275 | Loss: 4.2324
Batch 50/2275 | Loss: 3.9476
Batch 100/2275 | Loss: 3.6295
Batch 150/2275 | Loss: 3.8147
Batch 200/2275 | Loss: 3.4399
Batch 250/2275 | Loss: 3.3044
Batch 300/2275 | Loss: 3.2497
Batch 350/2275 | Loss: 3.3319
Batch 400/2275 | Loss: 3.4967
Batch 450/2275 | Loss: 2.9502
Batch 500/2275 | Loss: 3.2421
Batch 550/2275 | Loss: 3.0809
Batch 600/2275 | Loss: 3.3699
Batch 650/2275 | Loss: 2.9513
Batch 700/2275 | Loss: 3.9944
Batch 750/2275 | Loss: 2.5787
Batch 800/2275 | Loss: 3.0051
Batch 850/2275 | Loss: 3.3751
Batch 900/2275 | Loss: 3.2640
Batch 950/2275 | Loss: 3.6195
Batch 1000/2275 | Loss: 3.3017
Batch 1050/2275 | Loss: 3.5843
Batch 1100/2275 | Loss: 3.1172
Batch 1150/2275 | Loss: 2.7577
Batch 1200/2275 | Loss: 3.4651
Batch 1250/2275 | Loss: 3.0340
Batch 1300/2275 | Loss: 3.8182
Batch 1350/2275 | Loss: 2.7862
Batch 1400/2275 | Loss: 3.7206
Batch 1450/2275 | Loss: 3.4040
Batch 1500/2275 | Loss: 2.9441
Batch 1550/2275 | Loss: 3.0914
Batch 1600/2275 | Loss: 3.0045


In [38]:
baseline_results = {}

print("\n==============================")
print("🚀 BASELINE MODELS (CE LOSS)")
print("==============================")

for name in ["mobilenet", "vgg16", "vgg19"]:

    print(f"\n🔹 Training {name.upper()} (Baseline)")

    model = get_model(name)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    # BASELINE LOSS (ONLY CE)
    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.1
    ).to(DEVICE)

    for epoch in range(5):

        model.train()
        total_loss = 0
        preds_all, targets_all = [], []

        for i, (imgs, labels) in enumerate(train_loader):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            preds_all.extend(preds.cpu().numpy())
            targets_all.extend(labels.cpu().numpy())

            if i % 100 == 0:
                batch_acc = (preds == labels).float().mean().item()
                print(f"Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f} | Acc: {batch_acc:.4f}")

        train_acc = accuracy_score(targets_all, preds_all)

        val_acc, val_f1 = evaluate(model, val_loader)

        print(f"""
==============================
Epoch {epoch}
Train Loss: {total_loss/len(train_loader):.4f}
Train Accuracy: {train_acc:.4f}
Val Accuracy: {val_acc:.4f}
Val F1: {val_f1:.4f}
==============================
""")

    test_acc, test_f1 = evaluate(model, test_loader)
    baseline_results[name] = (test_acc, test_f1)

    print(f" BASELINE {name.upper()} → Acc: {test_acc:.4f}, F1: {test_f1:.4f}")


🚀 BASELINE MODELS (CE LOSS)

🔹 Training MOBILENET (Baseline)
Batch 0/1138 | Loss: 2.8430 | Acc: 0.0312
Batch 100/1138 | Loss: 2.8273 | Acc: 0.0312
Batch 200/1138 | Loss: 2.4739 | Acc: 0.1562
Batch 300/1138 | Loss: 2.4278 | Acc: 0.1562
Batch 400/1138 | Loss: 2.2799 | Acc: 0.3438
Batch 500/1138 | Loss: 2.3547 | Acc: 0.2812
Batch 600/1138 | Loss: 2.3065 | Acc: 0.2500
Batch 700/1138 | Loss: 2.3270 | Acc: 0.3438
Batch 800/1138 | Loss: 2.2617 | Acc: 0.2188
Batch 900/1138 | Loss: 2.0844 | Acc: 0.3125
Batch 1000/1138 | Loss: 2.2536 | Acc: 0.1875
Batch 1100/1138 | Loss: 2.1329 | Acc: 0.3750

Epoch 0
Train Loss: 2.3671
Train Accuracy: 0.2543
Val Accuracy: 0.2805
Val F1: 0.2908

Batch 0/1138 | Loss: 1.9883 | Acc: 0.2812
Batch 100/1138 | Loss: 1.9174 | Acc: 0.3125
Batch 200/1138 | Loss: 2.2220 | Acc: 0.4062
Batch 300/1138 | Loss: 2.3248 | Acc: 0.3125
Batch 400/1138 | Loss: 2.1248 | Acc: 0.3438
Batch 500/1138 | Loss: 2.1803 | Acc: 0.2500
Batch 600/1138 | Loss: 2.3236 | Acc: 0.1562
Batch 700/1138 |

In [39]:
print("\n📊 FINAL COMPARISON")

print("\n--- BASELINE (CE) ---")
for k, v in baseline_results.items():
    print(f"{k}: Acc={v[0]:.4f}, F1={v[1]:.4f}")

print("\n--- ORDINAL LOSS ---")
for k, v in results.items():
    print(f"{k}: Acc={v[0]:.4f}, F1={v[1]:.4f}")


📊 FINAL COMPARISON

--- BASELINE (CE) ---
mobilenet: Acc=0.3035, F1=0.3177
vgg16: Acc=0.2731, F1=0.2785
vgg19: Acc=0.2633, F1=0.2624

--- ORDINAL LOSS ---
mobilenet: Acc=0.3181, F1=0.3292
vgg16: Acc=0.2257, F1=0.2364
vgg19: Acc=0.2482, F1=0.2453


In [ ]:
from torchvision.models import inception_v3, Inception_V3_Weights

print("\n🚀 BASELINE INCEPTION (CE LOSS)")

# MODEL
model = inception_v3(weights=Inception_V3_Weights.DEFAULT)

model.fc = nn.Linear(model.fc.in_features, num_classes)
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)

model = model.to(DEVICE)
# OPTIMIZER
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# BASELINE LOSS
loss_fn = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
).to(DEVICE)

# TRAIN FUNCTION (INCEPTION)
def train_inception_ce(model, loader):
    model.train()
    total_loss = 0

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(imgs)

        # AUX HANDLING
        if isinstance(outputs, tuple):
            main_out, aux_out = outputs
            loss1 = loss_fn(main_out, labels)
            loss2 = loss_fn(aux_out, labels)
            loss = loss1 + 0.4 * loss2
        else:
            loss = loss_fn(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            preds = torch.argmax(main_out if isinstance(outputs, tuple) else outputs, dim=1)
            acc = (preds == labels).float().mean().item()
            print(f"Batch {i}/{len(loader)} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

    return total_loss / len(loader)

# EVALUATION (SAME AS BEFORE)
def eval_inception(model, loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)

            outputs = model(imgs)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            p = torch.argmax(outputs, dim=1).cpu().numpy()
            preds.extend(p)
            targets.extend(labels.numpy())

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="weighted")

    return acc, f1

# TRAIN LOOP
for epoch in range(5):
    loss = train_inception_ce(model, train_loader_inc)
    acc, f1 = eval_inception(model, val_loader_inc)

    print(f"""
==============================
Baseline Inception Epoch {epoch}
Loss: {loss:.4f}
Val Accuracy: {acc:.4f}
Val F1: {f1:.4f}
==============================
""")

# FINAL TEST
acc, f1 = eval_inception(model, test_loader_inc)

baseline_results["inception"] = (acc, f1)

print(f"\n BASELINE INCEPTION → Acc: {acc:.4f}, F1: {f1:.4f}")


🚀 BASELINE INCEPTION (CE LOSS)
Batch 0/2275 | Loss: 4.0236 | Acc: 0.0000
Batch 50/2275 | Loss: 3.8631 | Acc: 0.1875
Batch 100/2275 | Loss: 3.8626 | Acc: 0.0625
Batch 150/2275 | Loss: 3.5142 | Acc: 0.1875
Batch 200/2275 | Loss: 3.5230 | Acc: 0.1250
Batch 250/2275 | Loss: 3.6970 | Acc: 0.3125
Batch 300/2275 | Loss: 3.8222 | Acc: 0.1250
Batch 350/2275 | Loss: 3.2895 | Acc: 0.1250
Batch 400/2275 | Loss: 3.0657 | Acc: 0.3125
Batch 450/2275 | Loss: 3.4163 | Acc: 0.3125
Batch 500/2275 | Loss: 3.1018 | Acc: 0.2500
Batch 550/2275 | Loss: 3.3678 | Acc: 0.2500
Batch 600/2275 | Loss: 3.1232 | Acc: 0.3125
Batch 650/2275 | Loss: 3.0679 | Acc: 0.3125
Batch 700/2275 | Loss: 3.3761 | Acc: 0.1250
Batch 750/2275 | Loss: 3.0654 | Acc: 0.3750
Batch 800/2275 | Loss: 2.8409 | Acc: 0.2500
Batch 850/2275 | Loss: 3.6292 | Acc: 0.1250
Batch 900/2275 | Loss: 3.0423 | Acc: 0.2500
Batch 950/2275 | Loss: 2.8024 | Acc: 0.4375
Batch 1000/2275 | Loss: 3.2695 | Acc: 0.1875
Batch 1050/2275 | Loss: 2.9202 | Acc: 0.2500
B

In [41]:
print("\n📊 FINAL COMPARISON")

print("\n--- BASELINE (CE) ---")
for k, v in baseline_results.items():
    print(f"{k}: Acc={v[0]:.4f}, F1={v[1]:.4f}")

print("\n--- ORDINAL LOSS ---")
for k, v in results.items():
    print(f"{k}: Acc={v[0]:.4f}, F1={v[1]:.4f}")


📊 FINAL COMPARISON

--- BASELINE (CE) ---
mobilenet: Acc=0.3035, F1=0.3177
vgg16: Acc=0.2731, F1=0.2785
vgg19: Acc=0.2633, F1=0.2624
inception: Acc=0.3119, F1=0.3124

--- ORDINAL LOSS ---
mobilenet: Acc=0.3181, F1=0.3292
vgg16: Acc=0.2257, F1=0.2364
vgg19: Acc=0.2482, F1=0.2453
